# NYC Yellow Taxi Data Cleaning

## Objective

Apply the data-quality findings identified during the data-understanding stage and create task-specific analytical samples.

Rather than applying one universal cleaning rule, different subsets are created depending on the requirements of each downstream analysis:

- Citywide demand analysis
- Zone-level demand analysis
- Fare and trip-characteristic analysis

In [1]:
import pandas as pd
import numpy as np

file_path = "../data/raw/yellow_tripdata_2025-01.parquet"

df = pd.read_parquet(file_path)

df.shape

(3475226, 20)

## 1. Create Core Validation Features

Trip duration is reconstructed from pickup and drop-off timestamps.  
For demand analysis, fare, passenger count, and trip distance are not used as mandatory filters because they are not required to establish that a pickup occurred.

In [3]:
df["trip_duration_min"] = (
    df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
).dt.total_seconds() / 60


In [4]:
jan_start = pd.Timestamp("2025-01-01")
feb_start = pd.Timestamp("2025-02-01")

outside_jan = (df["tpep_pickup_datetime"] < jan_start) | (
    df["tpep_pickup_datetime"] >= feb_start
)

invalid_duration = df["trip_duration_min"] <= 0

extreme_duration = df["trip_duration_min"] > 360


In [5]:
cleaning_impact = pd.Series(
    {
        "outside_january": outside_jan.sum(),
        "nonpositive_duration": invalid_duration.sum(),
        "duration_over_6_hours": extreme_duration.sum(),
    }
)

cleaning_impact


outside_january            22
nonpositive_duration     2051
duration_over_6_hours    1203
dtype: int64

> A six-hour threshold is used to remove clearly implausible trip durations identified during manual inspection. This threshold is intended as a conservative data-quality filter rather than a statement that trips above six hours are impossible.

In [6]:
citywide_mask = ~outside_jan & ~invalid_duration & ~extreme_duration

citywide_demand = df.loc[citywide_mask].copy()


In [7]:
len(df), len(citywide_demand)


(3475226, 3471950)

In [8]:
len(citywide_demand) / len(df) * 100

99.90573274946722

## Sanity Check


In [9]:
(
    citywide_demand["tpep_pickup_datetime"].min(),
    citywide_demand["tpep_pickup_datetime"].max(),
)


(Timestamp('2025-01-01 00:00:00'), Timestamp('2025-01-31 23:59:59'))

In [10]:
citywide_demand["trip_duration_min"].describe()


count    3.471950e+06
mean     1.460748e+01
std      1.144391e+01
min      1.666667e-02
25%      7.283333e+00
50%      1.170000e+01
75%      1.833333e+01
max      3.599167e+02
Name: trip_duration_min, dtype: float64

In [14]:
(citywide_demand["trip_duration_min"] <= 0).sum()

(citywide_demand["trip_duration_min"] > 360).sum()

np.int64(0)

In [15]:
cleaning_impact


outside_january            22
nonpositive_duration     2051
duration_over_6_hours    1203
dtype: int64

In [16]:
len(df), len(citywide_demand)


(3475226, 3471950)

In [17]:
len(citywide_demand) / len(df) * 100


99.90573274946722

### 1.1 Citywide Demand Sample

For citywide demand analysis, only records with valid pickup timestamps and plausible trip durations are required.

The cleaning rules exclude:
- pickups outside January 2025,
- non-positive trip durations,
- trip durations greater than six hours.

The resulting sample retains over 99.9% of the original trip records. Fare, passenger count, and trip distance are not used as mandatory filters because they are not required to establish that a pickup occurred.

## 2. Zone-Level Demand Sample

Zone-level demand analysis requires both a valid trip record and a clearly identified NYC pickup zone.

Special location categories such as `Unknown` and `Outside of NYC` are excluded from zone-level analysis because they cannot be assigned to a specific NYC taxi zone.

In [19]:
zones = pd.read_csv("../data/raw/taxi_zone_lookup.csv")


In [20]:
pickup_zones = zones[["LocationID", "Borough", "Zone"]].rename(
    columns={
        "LocationID": "PULocationID",
        "Borough": "pickup_borough",
        "Zone": "pickup_zone",
    }
)


In [21]:
zone_demand = citywide_demand.merge(
    pickup_zones, on="PULocationID", how="left", validate="m:1"
)

len(zone_demand) == len(citywide_demand)

True

In [22]:
special_location = zone_demand["PULocationID"].isin([264, 265])


In [23]:
special_location.sum()


np.int64(9463)

In [24]:
zone_demand.loc[special_location, "PULocationID"].value_counts()


PULocationID
264    8089
265    1374
Name: count, dtype: int64

In [25]:
zone_demand = zone_demand.loc[~special_location].copy()


In [26]:
len(zone_demand)

len(zone_demand) / len(citywide_demand) * 100


99.72744423162776

In [27]:
zone_demand[["pickup_borough", "pickup_zone"]].isna().sum()


pickup_borough    0
pickup_zone       0
dtype: int64

### 2.1 Zone-Level Demand Sample

Starting from the cleaned citywide demand sample, trips with special or undefined pickup locations were excluded because they cannot be assigned to a specific NYC taxi zone.

The resulting zone-level sample retains approximately 99.7% of the citywide demand records, with no remaining missing values in `pickup_borough` or `pickup_zone`.

This sample will be used for all geographic demand analyses.

In [29]:
# Citywide demand time features
citywide_demand["pickup_date"] = citywide_demand["tpep_pickup_datetime"].dt.floor("D")

citywide_demand["pickup_hour"] = citywide_demand["tpep_pickup_datetime"].dt.hour

citywide_demand["pickup_hour_ts"] = citywide_demand["tpep_pickup_datetime"].dt.floor(
    "h"
)

citywide_demand["day_of_week"] = citywide_demand["tpep_pickup_datetime"].dt.day_name()

citywide_demand["is_weekend"] = (
    citywide_demand["tpep_pickup_datetime"].dt.dayofweek >= 5
)


## 3. Save Processed Analytical Samples

The cleaned citywide and zone-level demand samples are saved as Parquet files for downstream analysis.

Raw source files remain unchanged, while processed datasets are stored separately to ensure reproducibility.

In [30]:
from pathlib import Path

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)


In [31]:
citywide_cols = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_duration_min",
    "pickup_date",
    "pickup_hour",
    "pickup_hour_ts",
    "day_of_week",
    "is_weekend",
    "PULocationID",
]

citywide_demand[citywide_cols].to_parquet(
    processed_dir / "citywide_demand_2025_01.parquet", index=False, compression="zstd"
)


In [34]:
zone_demand["pickup_date"] = zone_demand["tpep_pickup_datetime"].dt.floor("D")

zone_demand["pickup_hour"] = zone_demand["tpep_pickup_datetime"].dt.hour

zone_demand["pickup_hour_ts"] = zone_demand["tpep_pickup_datetime"].dt.floor("h")

zone_demand["day_of_week"] = zone_demand["tpep_pickup_datetime"].dt.day_name()

zone_demand["is_weekend"] = zone_demand["tpep_pickup_datetime"].dt.dayofweek >= 5


In [35]:
zone_demand[
    [
        "tpep_pickup_datetime",
        "pickup_date",
        "pickup_hour",
        "pickup_hour_ts",
        "day_of_week",
        "is_weekend",
    ]
].head()


,tpep_pickup_datetime,pickup_date,pickup_hour,pickup_hour_ts,day_of_week,is_weekend
0,2025-01-01 00:18:38,2025-01-01,0,2025-01-01,Wednesday,False
1,2025-01-01 00:32:40,2025-01-01,0,2025-01-01,Wednesday,False
2,2025-01-01 00:44:04,2025-01-01,0,2025-01-01,Wednesday,False
3,2025-01-01 00:14:27,2025-01-01,0,2025-01-01,Wednesday,False
4,2025-01-01 00:21:34,2025-01-01,0,2025-01-01,Wednesday,False


In [36]:
zone_demand[zone_cols].to_parquet(
    processed_dir / "zone_demand_2025_01.parquet", index=False, compression="zstd"
)


In [37]:
citywide_check = pd.read_parquet("../data/processed/citywide_demand_2025_01.parquet")

zone_check = pd.read_parquet("../data/processed/zone_demand_2025_01.parquet")


In [38]:
citywide_check.shape, zone_check.shape


((3471950, 9), (3462487, 9))

In [39]:
zone_check[["pickup_borough", "pickup_zone"]].isna().sum()


pickup_borough    0
pickup_zone       0
dtype: int64

In [40]:
citywide_check["pickup_date"].min(), citywide_check["pickup_date"].max()


(Timestamp('2025-01-01 00:00:00'), Timestamp('2025-01-31 00:00:00'))

## 4. Final Processed Samples

Two analytical datasets were created:

- **Citywide demand sample:** 3,471,950 trips with valid January 2025 pickup timestamps and plausible trip durations.
- **Zone-level demand sample:** 3,462,487 trips with valid citywide demand records and identifiable NYC pickup zones.

Both datasets were successfully saved and reloaded from Parquet files, confirming that the processed outputs are ready for downstream demand analysis.